In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import numpy as np
import os


class BiomassDataset(Dataset):

    def __init__(
        self,
        csv_path: str,
        image_root: str = "",
        transform=None,
        is_train: bool = True,
    ):
        self.df = pd.read_csv(csv_path)
        self.image_root = image_root
        self.transform = transform
        self.is_train = is_train


        self.numeric_cols = [
            "ndvi_norm",
            "height_norm",
            "date_sin",
            "date_cos",
            "date_sin2",
            "date_cos2",
            "ndvi_height_interaction",
            "seasonal_ndvi",
        ]

        self.categorical_cols = [
            "state_id",
            "species_id",
        ]

        self.target_col = "target"

        # ---- NUMERIC PREPROCESSING ----
        for col in self.numeric_cols:
            # Force numeric + float32
            self.df[col] = (
                pd.to_numeric(self.df[col], errors="coerce")
                .astype(np.float32)
            )

    
            if self.df[col].isnull().any():
                self.df[col].fillna(self.df[col].mean(), inplace=True)

  
        for col in self.categorical_cols:
            
            if self.df[col].isnull().any():
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)

 
            self.df[col] = self.df[col].astype(np.int64)

  
        if self.is_train:
            self.df[self.target_col] = (
                pd.to_numeric(self.df[self.target_col], errors="coerce")
                .astype(np.float32)
            )

    def __len__(self):
        return len(self.df)

    def _load_image(self, image_path: str):
        full_path = os.path.join(self.image_root, image_path)
        image = Image.open(full_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

    def __getitem__(self, idx):
  
        image_path = self.df.loc[idx, "image_path"]
        image = self._load_image(image_path)

  
        numeric = torch.from_numpy(
            self.df.loc[idx, self.numeric_cols]
            .to_numpy(dtype=np.float32)
        )

        categorical = torch.from_numpy(
            self.df.loc[idx, self.categorical_cols]
            .to_numpy(dtype=np.int64)
        )

        if self.is_train:
            target = torch.tensor(
                self.df.loc[idx, self.target_col],
                dtype=torch.float32
            )
            return image, numeric, categorical, target

        return image, numeric, categorical


In [21]:
ds = BiomassDataset(
    csv_path=r"C:\Users\StrucData 1\Downloads\train_processed.csv",
    image_root=r"C:\Users\StrucData 1\Downloads\csiro-biomass",
    transform=None,
    is_train=True
)

img, num, cat, y = ds[0]

print(num.dtype, num.shape)
print(cat.dtype, cat.shape)
print(y)


torch.float32 torch.Size([8])
torch.int64 torch.Size([2])
tensor(0.)
